# Image Classification ด้วย HuggingFace: เปรียบเทียบ CNN กับ Vision Transformer

ใน notebook นี้คุณจะได้เรียนรู้ workflow การทำ image classification แบบ end-to-end ด้วย HuggingFace ecosystem โดยจะเปรียบเทียบ pre-trained models 2 ตัว:

- **ResNet-18** (CNN architecture)
- **DeiT-tiny** (Vision Transformer architecture)

เราจะใช้ **Oxford-IIIT Pet Dataset** ซึ่งมี 37 classes ของพันธุ์สุนัขและแมว ประมาณ 7,400 รูป

**เป้าหมายการเรียนรู้:**
- ใช้ `transformers`, `datasets`, `evaluate` libraries
- สร้าง HuggingFace dataset จาก CSV metadata
- Fine-tune pre-trained models ด้วย `Trainer` API
- Evaluate และเปรียบเทียบ performance ระหว่าง CNN กับ ViT

## 1. Setup - ติดตั้ง Libraries และตรวจสอบ GPU

ก่อนเริ่ม ต้องติดตั้ง libraries ที่จำเป็นและตรวจสอบว่ามี GPU ให้ใช้หรือไม่

In [ ]:
# ติดตั้ง libraries ที่จำเป็น
!pip install transformers datasets evaluate torch torchvision Pillow pandas matplotlib seaborn scikit-learn -q

In [ ]:
# Import libraries ทั้งหมดที่ใช้
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torchvision
from torchvision.transforms import RandomHorizontalFlip, ColorJitter, Compose
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# HuggingFace libraries
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset, Dataset, ClassLabel
import evaluate

# ตั้งค่า random seed เพื่อ reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ตรวจสอบ GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected. Training will be slow!')

## 2. เตรียม Dataset - ดาวน์โหลด Oxford-IIIT Pet Dataset

เราจะดาวน์โหลด Oxford-IIIT Pet Dataset ผ่าน `torchvision` และสร้าง CSV metadata file

**ทำไมต้องสร้าง CSV?**
ในงานจริง คุณมักจะมี data อยู่ในรูปแบบ files พร้อม metadata แยกต่างหาก การสร้าง CSV และโหลดด้วย HuggingFace `load_dataset` คือ skill ที่สำคัญมาก

In [ ]:
# สร้าง directory สำหรับเก็บ data
!mkdir -p data/images

# ดาวน์โหลด Oxford-IIIT Pet Dataset
# ใช้ torchvision เพื่อดาวน์โหลดและ extract
from torchvision.datasets import OxfordIIITPet

# Download dataset (ครั้งแรกจะใช้เวลาสักครู่)
dataset_path = './data'
oxford_dataset = OxfordIIITPet(
    root=dataset_path,
    download=True,
    target_types='category'  # 37 categories เท่านั้น
)

print(f'Dataset size: {len(oxford_dataset)} images')
print(f'Number of classes: {len(oxford_dataset.classes)}')
print(f'Class names (first 10): {oxford_dataset.classes[:10]}')

### สร้าง Metadata CSV

ตอนนี้เราจะสร้างไฟล์ `metadata.csv` ที่มี structure ดังนี้:
```
file_name,label
images/Abyssinian_1.jpg,Abyssinian
```

ซึ่งเป็น format ที่ใช้บ่อยในงานจริง

In [ ]:
# สร้าง metadata.csv
metadata_data = []

# OxfordIIITPet เก็บรูปใน images/ และ annotations/ โฟลเดอร์
from torchvision.datasets.utils import download_and_extract_archive
import os
import shutil

# คัดลอกไฟล์รูปภาพไปยัง data/images/
source_images_dir = os.path.join(dataset_path, 'oxford-iiit-pet', 'images')
target_images_dir = os.path.join(dataset_path, 'images')

if os.path.exists(source_images_dir):
    # คัดลอกไฟล์ทั้งหมดไป data/images/
    if not os.path.exists(target_images_dir):
        os.makedirs(target_images_dir)

    for filename in os.listdir(source_images_dir):
        if filename.endswith('.jpg'):
            shutil.copy(
                os.path.join(source_images_dir, filename),
                os.path.join(target_images_dir, filename)
            )

    print(f'Copied {len([f for f in os.listdir(target_images_dir) if f.endswith(".jpg")])} images')

# สร้าง metadata.csv
class_to_idx = oxford_dataset.class_to_idx

for idx in range(len(oxford_dataset)):
    # OxfordIIITPet returns (image, label_index)
    _, label_idx = oxford_dataset[idx]

    # หาชื่อไฟล์ (ตาม naming convention ของ Oxford Pet)
    # Format: class_name_index.jpg
    class_name = oxford_dataset.classes[label_idx]

    # หาไฟล์ที่เกี่ยวข้อง (มีไฟล์ .jpg หลายไฟล์ต่อ class)
    # เราจะ iterate ผ่านไฟล์ทั้งหมดใน target_images_dir
    pass  # เราจะสร้าง metadata ในวิธีอื่นที่ง่ายกว่า

print('Preparing metadata from file structure...')

In [ ]:
# สร้าง metadata จาก file structure โดยตรง (ง่ายกว่า)
metadata_data = []

for filename in sorted(os.listdir(target_images_dir)):
    if filename.endswith('.jpg'):
        # Extract class name from filename
        # Format: Abyssinian_1.jpg, Abyssian_2.jpg, etc.
        # บางไฟล์มีเลขต่อท้าย บางไฟล์ไม่มี
        class_name = '_'.join(filename.split('_')[:-1])

        # ตรวจสอบว่า class_name อยู่ใน class list หรือไม่
        if class_name in class_to_idx:
            metadata_data.append({
                'file_name': f'images/{filename}',
                'label': class_name
            })

# สร้าง DataFrame และบันทึกเป็น CSV
metadata_df = pd.DataFrame(metadata_data)
metadata_df.to_csv(os.path.join(dataset_path, 'metadata.csv'), index=False)

print(f'Created metadata.csv with {len(metadata_df)} entries')
print(f'\nFirst 5 entries:')
print(metadata_df.head())
print(f'\nUnique labels: {metadata_df["label"].nunique()}')
print(f'\nLabel distribution (first 10):')
print(metadata_df['label'].value_counts().head(10))

In [ ]:
# แสดงตัวอย่างรูปภาพ
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.ravel()

# สุ่มเลือก 8 รูป
sample_indices = np.random.choice(len(metadata_df), 8, replace=False)

for idx, ax in zip(sample_indices, axes):
    row = metadata_df.iloc[idx]
    img_path = os.path.join(dataset_path, row['file_name'])
    img = Image.open(img_path)

    ax.imshow(img)
    ax.set_title(f"{row['label']}")
    ax.axis('off')

plt.tight_layout()
plt.show()
print('Sample images from dataset')

### Split Dataset: Train / Validation / Test

เราจะแบ่งข้อมูลเป็นสามส่วน:
- **Train (70%)**: สำหรับ fine-tune models
- **Validation (15%)**: สำหรับ tune hyperparameters และ monitor training
- **Test (15%)**: สำหรับ final evaluation เปรียบเทียบ models

In [ ]:
# Split dataset เป็น train/val/test = 70/15/15
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    metadata_df,
    test_size=0.3,
    stratify=metadata_df['label'],  # Stratify เพื่อให้ class distribution สมดุล
    random_state=42
)

# Second split: 15% val, 15% test (จาก 30% temp)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,  # 50% ของ temp คือ 15% ของทั้งหมด
    stratify=temp_df['label'],
    random_state=42
)

print(f'Train set: {len(train_df)} images ({len(train_df)/len(metadata_df)*100:.1f}%)')
print(f'Validation set: {len(val_df)} images ({len(val_df)/len(metadata_df)*100:.1f}%)')
print(f'Test set: {len(test_df)} images ({len(test_df)/len(metadata_df)*100:.1f}%)')

# บันทึกแต่ละ split เป็น CSV แยก
train_df.to_csv(os.path.join(dataset_path, 'train_metadata.csv'), index=False)
val_df.to_csv(os.path.join(dataset_path, 'val_metadata.csv'), index=False)
test_df.to_csv(os.path.join(dataset_path, 'test_metadata.csv'), index=False)

print('\nSaved metadata files for each split')

## 3. โหลด Dataset ด้วย HuggingFace Datasets

ตอนนี้เราจะใช้ `load_dataset` โหลด CSV metadata และแปลงให้อยู่ในรูปแบบ HuggingFace Dataset

**Key concept:** `ClassLabel` feature ช่วยแปลง string labels → integers อัตโนมัติ

In [ ]:
# โหลด datasets จาก CSV files
dataset_dict = load_dataset(
    'csv',
    data_files={
        'train': os.path.join(dataset_path, 'train_metadata.csv'),
        'validation': os.path.join(dataset_path, 'val_metadata.csv'),
        'test': os.path.join(dataset_path, 'test_metadata.csv')
    }
)

print('Dataset loaded successfully!')
print(f'\nDataset structure:')
print(dataset_dict)

print(f'\nTrain set features:')
print(dataset_dict['train'].features)

In [ ]:
# แปลง labels จาก strings → integers ด้วย ClassLabel
# ClassLabel คือ feature type พิเศษใน HuggingFace ที่จัดการ label encoding

# สร้าง ClassLabel feature จาก unique labels
label_list = list(metadata_df['label'].unique())
class_label = ClassLabel(num_classes=len(label_list), names=label_list)

# Cast label column เป็น ClassLabel
dataset_dict = dataset_dict.cast_column('label', class_label)

print('Labels converted to integers!')
print(f'\nClassLabel feature:')
print(dataset_dict['train'].features['label'])

print(f'\nLabel mapping (first 5):')
for i in range(min(5, len(label_list))):
    print(f'  {i} → {label_list[i]}')

In [ ]:
# แปลง file_name เป็น full path และโหลดรูปภาพ
def load_image_from_path(example):
    """Load image from file path and add to dataset"""
    img_path = os.path.join(dataset_path, example['file_name'])
    image = Image.open(img_path).convert('RGB')  # แปลงเป็น RGB เพื่อความสม่ำเสมอ
    example['image'] = image
    return example

# ใช้ map ใช้ load_image function กับทุก example
dataset_dict = dataset_dict.map(load_image_from_path, num_proc=1)

# ลบ column file_name ทิ้ง (ไม่ต้องการแล้ว)
dataset_dict = dataset_dict.remove_columns(['file_name'])

print('Images loaded into dataset!')
print(f'\nDataset columns: {dataset_dict["train"].column_names}')
print(f'\nFirst example:')
print(dataset_dict['train'][0])

In [ ]:
# ตรวจสอบ label distribution ในแต่ละ split
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (split_name, split_data) in enumerate(dataset_dict.items()):
    label_counts = pd.Series(split_data['label']).value_counts().sort_index()
    axes[idx].bar(range(len(label_counts)), label_counts.values)
    axes[idx].set_title(f'{split_name.capitalize()} Set Distribution')
    axes[idx].set_xlabel('Class Index')
    axes[idx].set_ylabel('Count')

plt.tight_layout()
plt.show()
print('Label distribution across splits (should be similar)')

## 4. Preprocessing - การเตรียม Data สำหรับ Models

แต่ละ model มี preprocessing requirements ที่ต่างกัน (size, normalization) เราจะใช้ `AutoImageProcessor` เพื่อโหลด processor ที่เหมาะสมกับแต่ละ model

**ทำไมต้องใช้ AutoImageProcessor?**
- แต่ละ model train มาพร้อม specific preprocessing (mean, std, resize strategy)
- ใช้ processor ที่ถูกต้องจะได้ performance ที่ดีที่สุด
- `Auto*` classes ทำให้ code ของเรา generic และใช้ได้กับทุก model

In [ ]:
# โหลด image processors สำหรับทั้งสอง models
resnet_processor = AutoImageProcessor.from_pretrained('microsoft/resnet-18')
deit_processor = AutoImageProcessor.from_pretrained('facebook/deit-tiny-patch16-224')

print('ResNet-18 processor:')
print(f'  Image size: {resnet_processor.size}')
print(f'  Mean: {resnet_processor.image_mean}')
print(f'  Std: {resnet_processor.image_std}')

print('\nDeiT-tiny processor:')
print(f'  Image size: {deit_processor.size}')
print(f'  Mean: {deit_processor.image_mean}')
print(f'  Std: {deit_processor.image_std}')

### สร้าง Transformation Functions

เราจะสร้าง functions สำหรับ:
1. **Train**: Apply preprocessing + augmentation (RandomHorizontalFlip, ColorJitter)
2. **Validation/Test**: Apply preprocessing เท่านั้น

**Important:** Augmentation ใช้กับ train set เท่านั้น เพื่อให้ evaluation แม่นยำ

In [ ]:
# สร้าง augmentation transforms สำหรับ training
train_augmentation = Compose([
    RandomHorizontalFlip(p=0.5),  # พลิกรูปแนวนอน 50%
    ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # ปรับสี
])

def preprocess_train(example, processor):
    """Apply augmentation + preprocessing for training"""
    # Apply augmentation
    image = train_augmentation(example['image'])

    # Apply model-specific preprocessing
    # processor จะทำการ resize, normalize ให้อัตโนมัติ
    inputs = processor(images=image, return_tensors='pt')

    # processor คืนค่า pixel_values มา ซึ่งมี shape [1, C, H, W]
    # เราต้อง squeeze เพื่อให้เป็น [C, H, W]
    example['pixel_values'] = inputs['pixel_values'].squeeze(0)

    return example

def preprocess_val(example, processor):
    """Apply only preprocessing (no augmentation) for validation/test"""
    # Apply model-specific preprocessing เท่านั้น
    inputs = processor(images=example['image'], return_tensors='pt')
    example['pixel_values'] = inputs['pixel_values'].squeeze(0)

    return example

print('Preprocessing functions created!')

In [ ]:
# ทดสอบ preprocessing
sample_image = dataset_dict['train'][0]['image']
print(f'Original image size: {sample_image.size}')

# Test ResNet preprocessing
resnet_inputs = preprocess_train(dataset_dict['train'][0], resnet_processor)
print(f'ResNet processed shape: {resnet_inputs["pixel_values"].shape}')

# Test DeiT preprocessing
deit_inputs = preprocess_train(dataset_dict['train'][0], deit_processor)
print(f'DeiT processed shape: {deit_inputs["pixel_values"].shape}')

# Visualize ผลของ augmentation
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()

for i in range(3):
    # Original
    axes[i*2].imshow(sample_image)
    axes[i*2].set_title('Original')
    axes[i*2].axis('off')

    # Augmented
    aug_image = train_augmentation(sample_image)
    axes[i*2+1].imshow(aug_image)
    axes[i*2+1].set_title('Augmented')
    axes[i*2+1].axis('off')

plt.tight_layout()
plt.show()
print('\nAugmentation examples (different each time due to randomness)')

### Apply Preprocessing ไปยัง Datasets

ตอนนี้เราจะใช้ preprocessing functions กับ datasets ทั้งหมด

In [ ]:
# Apply preprocessing สำหรับ ResNet-18
resnet_train_dataset = dataset_dict['train'].map(
    lambda x: preprocess_train(x, resnet_processor),
    batched=False,
    num_proc=1
)
resnet_val_dataset = dataset_dict['validation'].map(
    lambda x: preprocess_val(x, resnet_processor),
    batched=False,
    num_proc=1
)
resnet_test_dataset = dataset_dict['test'].map(
    lambda x: preprocess_val(x, resnet_processor),
    batched=False,
    num_proc=1
)

print('ResNet-18 datasets preprocessed!')
print(f'Train set columns: {resnet_train_dataset.column_names}')
print(f'Train set size: {len(resnet_train_dataset)}')

In [ ]:
# Apply preprocessing สำหรับ DeiT-tiny
deit_train_dataset = dataset_dict['train'].map(
    lambda x: preprocess_train(x, deit_processor),
    batched=False,
    num_proc=1
)
deit_val_dataset = dataset_dict['validation'].map(
    lambda x: preprocess_val(x, deit_processor),
    batched=False,
    num_proc=1
)
deit_test_dataset = dataset_dict['test'].map(
    lambda x: preprocess_val(x, deit_processor),
    batched=False,
    num_proc=1
)

print('DeiT-tiny datasets preprocessed!')
print(f'Train set columns: {deit_train_dataset.column_names}')
print(f'Train set size: {len(deit_train_dataset)}')

In [ ]:
# ลบ image column ทิ้ง (เพื่อประหยัด memory — เรามี pixel_values แล้ว)
# **สำคัญ**: ต้องลบ image column ออกจาก datasets ที่จะใช้สำหรับ training/evaluation
# เพราะ image sizes ต่างกันจะทำให้ dataloader ผิดพลาดเมื่อทำ batching

# Set format เป็น torch เพื่อให้ใช้กับ PyTorch ได้
# สำหรับ training/evaluation: เก็บเฉพาะ pixel_values และ label
resnet_train_dataset.set_format('torch', columns=['pixel_values', 'label'])
resnet_val_dataset.set_format('torch', columns=['pixel_values', 'label'])
resnet_test_dataset.set_format('torch', columns=['pixel_values', 'label'])

deit_train_dataset.set_format('torch', columns=['pixel_values', 'label'])
deit_val_dataset.set_format('torch', columns=['pixel_values', 'label'])
deit_test_dataset.set_format('torch', columns=['pixel_values', 'label'])

print('Datasets ready for training!')
print('Note: image column removed to avoid batching issues during evaluation')


## 5. Fine-tune Model 1: ResNet-18

ตอนนี้เราจะ fine-tune ResNet-18 บน Oxford Pet dataset

**Key concepts:**
- `AutoModelForImageClassification`: โหลด model architecture อัตโนมัติ
- ปรับ output layer ให้มี 37 classes (แทนที่ ImageNet 1000 classes)
- `TrainingArguments`: ตั้งค่า hyperparameters
- `Trainer`: HuggingFace training loop ที่สะดวกและมีประสิทธิภาพ

In [ ]:
# โหลด ResNet-18 model
model_id = 'microsoft/resnet-18'

# โหลด model พร้อมปรับ classifier head
# AutoModelForImageClassification จะ detect label names และปรับ output layer อัตโนมัติ
resnet_model = AutoModelForImageClassification.from_pretrained(
    model_id,
    num_labels=len(label_list),  # 37 classes
    id2label={i: label for i, label in enumerate(label_list)},
    label2id={label: i for i, label in enumerate(label_list)},
    ignore_mismatched_sizes=True  # ให้ resize classifier layer อัตโนมัติ
)

print('ResNet-18 model loaded!')
print(f'Model architecture:\n{resnet_model}')
print(f'\nTotal parameters: {sum(p.numel() for p in resnet_model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in resnet_model.parameters() if p.requires_grad):,}')

In [ ]:
# ตรวจสอบ classifier head ที่ถูกปรับ
print('Classifier head:')
print(f'  Type: {type(resnet_model.classifier)}')
if hasattr(resnet_model.classifier, 'out_features'):
    print(f'  Output features: {resnet_model.classifier.out_features}')
elif hasattr(resnet_model.classifier, 'out_channels'):
    print(f'  Output channels: {resnet_model.classifier.out_channels}')

### สร้าง Compute Metrics Function

`compute_metrics` คือ function ที่ `Trainer` จะเรียกหลังจาก每一 epoch evaluation
จะรับ `EvalPrediction` (predictions + labels) และคืนค่า metrics เป็น dict

In [ ]:
# โหลด metrics ที่ต้องการ
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

def compute_metrics(eval_pred):
    """Compute metrics for evaluation"""
    logits, labels = eval_pred

    # แปลง logits → predictions โดยหา index ที่มีค่าสูงสุด
    predictions = logits.argmax(axis=-1)

    # คำนวณ metrics
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)['accuracy']
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average='macro')['f1']
    f1_weighted = f1_metric.compute(predictions=predictions, references=labels, average='weighted')['f1']
    precision_macro = precision_metric.compute(predictions=predictions, references=labels, average='macro', zero_division=0)['precision']
    recall_macro = recall_metric.compute(predictions=predictions, references=labels, average='macro', zero_division=0)['recall']

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
    }

print('Metrics function created!')

### ตั้งค่า Training Arguments และเริ่ม Training

**Training hyperparameters สำหรับ Colab Free Tier:**
- `batch_size=32`: ขนาดที่ reasonable สำหรับ T4 GPU 16GB
- `num_train_epochs=4`: เพียงพอสำหรับ fine-tuning
- `fp16=True`: Mixed precision training เพื่อความเร็วและประหยัด memory
- `learning_rate=5e-5`: Learning rate ต่ำสำหรับ fine-tuning
- `weight_decay=0.01`: Regularization

In [ ]:
# ตั้งค่า TrainingArguments
training_args = TrainingArguments(
    output_dir='./results/resnet18',

    # Training hyperparameters
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,

    # Optimizer
    optim='adamw_torch',  # AdamW optimizer

    # Mixed precision training (เร็วขึ้นและประหยัด memory)
    fp16=True,

    # Evaluation & logging
    eval_strategy='epoch',  # ประเมินทุก epoch
    logging_strategy='epoch',
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=1,  # เก็บ checkpoint ล่าสุดเท่านั้น

    # Data loading
    dataloader_num_workers=2,

    # Disable wandb/tensorboard logging
    report_to='none',

    # Misc
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    remove_unused_columns=False,  # สำคัญ! เพื่อเก็บ pixel_values
)

print('Training arguments configured!')

In [ ]:
# สร้าง Trainer
resnet_trainer = Trainer(
    model=resnet_model,
    args=training_args,
    train_dataset=resnet_train_dataset,
    eval_dataset=resnet_val_dataset,
    compute_metrics=compute_metrics,
)

print('Trainer created for ResNet-18!')

In [ ]:
# เริ่ม training (จะใช้เวลา ~10-15 นาที บน Colab T4)
import time

print('Starting ResNet-18 training...')
print('This will take ~10-15 minutes on Colab T4 GPU\n')

start_time = time.time()
resnet_train_result = resnet_trainer.train()
resnet_training_time = time.time() - start_time

print(f'\nTraining completed in {resnet_training_time/60:.1f} minutes!')
print(f'\nFinal training metrics:')
for key, value in resnet_train_result.metrics.items():
    print(f'  {key}: {value:.4f}')

In [ ]:
# Plot training history
if hasattr(resnet_trainer, 'state') and hasattr(resnet_trainer.state, 'log_history'):
    logs = resnet_trainer.state.log_history

    # แยก train และ eval logs
    train_logs = [log for log in logs if 'loss' in log and 'epoch' in log]
    eval_logs = [log for log in logs if 'eval_loss' in log]

    if train_logs and eval_logs:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Loss curve
        epochs_train = [log['epoch'] for log in train_logs]
        losses_train = [log['loss'] for log in train_logs]
        axes[0].plot(epochs_train, losses_train, label='Train Loss')

        epochs_eval = [log['epoch'] for log in eval_logs]
        losses_eval = [log['eval_loss'] for log in eval_logs]
        axes[0].plot(epochs_eval, losses_eval, label='Val Loss', marker='o')

        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('ResNet-18 Training Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Accuracy curve
        eval_acc = [log['eval_accuracy'] for log in eval_logs]
        axes[1].plot(epochs_eval, eval_acc, label='Val Accuracy', marker='o', color='green')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].set_title('ResNet-18 Validation Accuracy')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

In [ ]:
# ประเมินบน validation set อีกครั้งเพื่อดู metrics ทั้งหมด
resnet_val_metrics = resnet_trainer.evaluate(resnet_val_dataset)

print('ResNet-18 Validation Set Metrics:')
print('-' * 50)
for metric, value in resnet_val_metrics.items():
    if metric.startswith('eval_'):
        print(f'{metric.replace("eval_", "").capitalize():20s}: {value:.4f}')

## 6. Fine-tune Model 2: DeiT-tiny (Vision Transformer)

ตอนนี้เราจะ fine-tune DeiT-tiny ซึ่งเป็น Vision Transformer architecture

**สำคัญ:** เราจะใช้ pipeline เหมือนเดิมทุกประการ เพื่อให้เปรียบเทียบได้อย่าง fair

In [ ]:
# โหลด DeiT-tiny model
model_id = 'facebook/deit-tiny-patch16-224'

deit_model = AutoModelForImageClassification.from_pretrained(
    model_id,
    num_labels=len(label_list),
    id2label={i: label for i, label in enumerate(label_list)},
    label2id={label: i for i, label in enumerate(label_list)},
    ignore_mismatched_sizes=True
)

print('DeiT-tiny model loaded!')
print(f'Model architecture:\n{deit_model}')
print(f'\nTotal parameters: {sum(p.numel() for p in deit_model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in deit_model.parameters() if p.requires_grad):,}')

In [ ]:
# ใช้ TrainingArguments เหมือนกับ ResNet-18 (เพื่อ fair comparison)
deit_training_args = TrainingArguments(
    output_dir='./results/deit_tiny',

    # เหมือนกับ ResNet-18 ทุกประการ
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    optim='adamw_torch',
    fp16=True,
    eval_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    dataloader_num_workers=2,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    remove_unused_columns=False,
)

print('Training arguments configured for DeiT-tiny!')

In [ ]:
# สร้าง Trainer สำหรับ DeiT-tiny
deit_trainer = Trainer(
    model=deit_model,
    args=deit_training_args,
    train_dataset=deit_train_dataset,
    eval_dataset=deit_val_dataset,
    compute_metrics=compute_metrics,
)

print('Trainer created for DeiT-tiny!')

In [ ]:
# เริ่ม training
print('Starting DeiT-tiny training...')
print('This will take ~15-20 minutes on Colab T4 GPU\n')

start_time = time.time()
deit_train_result = deit_trainer.train()
deit_training_time = time.time() - start_time

print(f'\nTraining completed in {deit_training_time/60:.1f} minutes!')
print(f'\nFinal training metrics:')
for key, value in deit_train_result.metrics.items():
    print(f'  {key}: {value:.4f}')

In [ ]:
# Plot training history
if hasattr(deit_trainer, 'state') and hasattr(deit_trainer.state, 'log_history'):
    logs = deit_trainer.state.log_history

    train_logs = [log for log in logs if 'loss' in log and 'epoch' in log]
    eval_logs = [log for log in logs if 'eval_loss' in log]

    if train_logs and eval_logs:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Loss curve
        epochs_train = [log['epoch'] for log in train_logs]
        losses_train = [log['loss'] for log in train_logs]
        axes[0].plot(epochs_train, losses_train, label='Train Loss')

        epochs_eval = [log['epoch'] for log in eval_logs]
        losses_eval = [log['eval_loss'] for log in eval_logs]
        axes[0].plot(epochs_eval, losses_eval, label='Val Loss', marker='o')

        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('DeiT-tiny Training Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Accuracy curve
        eval_acc = [log['eval_accuracy'] for log in eval_logs]
        axes[1].plot(epochs_eval, eval_acc, label='Val Accuracy', marker='o', color='green')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].set_title('DeiT-tiny Validation Accuracy')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

In [ ]:
# ประเมินบน validation set
deit_val_metrics = deit_trainer.evaluate(deit_val_dataset)

print('DeiT-tiny Validation Set Metrics:')
print('-' * 50)
for metric, value in deit_val_metrics.items():
    if metric.startswith('eval_'):
        print(f'{metric.replace("eval_", "").capitalize():20s}: {value:.4f}')

In [ ]:
# สร้าง dataset พิเศษสำหรับ visualization ที่เก็บ image ไว้
# (สำหรับ qualitative analysis ในภายหลัง)

# โหลด test metadata เพื่อใช้ในการโหลดรูปภาพ
test_df = pd.read_csv(os.path.join(dataset_path, 'test_metadata.csv'))

# สร้าง visualization data จาก test metadata
resnet_test_viz = []
for idx in range(len(test_df)):
    row = test_df.iloc[idx]
    img_path = os.path.join(dataset_path, row['file_name'])
    image = Image.open(img_path).convert('RGB')

    # หา label index
    label_idx = label_list.index(row['label'])

    resnet_test_viz.append({
        'image': image,
        'label': label_idx
    })

# DeiT ใช้ test set เดียวกัน
deit_test_viz = resnet_test_viz

print(f'Created visualization datasets with {len(resnet_test_viz)} samples')

## 7. Evaluation - เปรียบเทียบทั้งสอง Models บน Test Set

ตอนนี้เราจะประเมินทั้งสอง models บน test set ที่ไม่เคยเห็นมาก่อน และวิเคราะห์ผลลัพธ์อย่างละเอียด

In [ ]:
# ประเมิน ResNet-18 บน test set
print('Evaluating ResNet-18 on test set...')
resnet_test_metrics = resnet_trainer.evaluate(resnet_test_dataset, metric_key_prefix='test')

print('\nResNet-18 Test Set Metrics:')
print('-' * 50)
for metric, value in resnet_test_metrics.items():
    if metric.startswith('test_'):
        print(f'{metric.replace("test_", "").capitalize():20s}: {value:.4f}')

In [ ]:
# ประเมิน DeiT-tiny บน test set
print('Evaluating DeiT-tiny on test set...')
deit_test_metrics = deit_trainer.evaluate(deit_test_dataset, metric_key_prefix='test')

print('\nDeiT-tiny Test Set Metrics:')
print('-' * 50)
for metric, value in deit_test_metrics.items():
    if metric.startswith('test_'):
        print(f'{metric.replace("test_", "").capitalize():20s}: {value:.4f}')

### สร้าง Confusion Matrices

Confusion matrix ช่วยให้เห็นว่า model สับสนระหว่าง classes ไหนบ้าง

In [ ]:
# ฟังก์ชันสร้าง predictions
def get_predictions(trainer, dataset):
    """Get predictions from trainer"""
    predictions = trainer.predict(dataset)
    y_pred = predictions.predictions.argmax(axis=-1)
    y_true = predictions.label_ids
    return y_pred, y_true

# Get predictions สำหรับทั้งสอง models
resnet_y_pred, resnet_y_true = get_predictions(resnet_trainer, resnet_test_dataset)
deit_y_pred, deit_y_true = get_predictions(deit_trainer, deit_test_dataset)

print('Predictions generated!')

In [ ]:
# สร้าง confusion matrices
resnet_cm = confusion_matrix(resnet_y_true, resnet_y_pred)
deit_cm = confusion_matrix(deit_y_true, deit_y_pred)

# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ResNet-18
sns.heatmap(resnet_cm, annot=False, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('ResNet-18 Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# DeiT-tiny
sns.heatmap(deit_cm, annot=False, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('DeiT-tiny Confusion Matrix')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()

print('Confusion matrices (darker = more predictions)')

### วิเคราะห์ Confusion Matrices

เราจะดูว่า classes ไหนที่ model ทำนายผิดบ่อยที่สุด

In [ ]:
# หา top 5 pairs ที่ model สับสนบ่อยที่สุด (ไม่รวม diagonal)
def get_most_confused(cm, label_names, top_n=5):
    """Get the most confused class pairs"""
    # สร้าง copy และ set diagonal เป็น 0 (ไม่นับ correct predictions)
    cm_no_diag = cm.copy()
    np.fill_diagonal(cm_no_diag, 0)

    # หา indices ที่มีค่าสูงสุด
    flat_indices = np.argpartition(cm_no_diag.ravel(), -top_n)[-top_n:]
    flat_indices = flat_indices[np.argsort(-cm_no_diag.ravel()[flat_indices])]

    confused_pairs = []
    for idx in flat_indices:
        i, j = np.unravel_index(idx, cm.shape)
        if cm_no_diag[i, j] > 0:  # เฉพาะที่มีค่า > 0
            confused_pairs.append({
                'true_label': label_names[i],
                'pred_label': label_names[j],
                'count': cm[i, j]
            })

    return confused_pairs[:top_n]

resnet_confused = get_most_confused(resnet_cm, label_list, top_n=5)
deit_confused = get_most_confused(deit_cm, label_list, top_n=5)

print('ResNet-18 - Top 5 Most Confused Pairs:')
print('-' * 60)
for pair in resnet_confused:
    print(f"{pair['true_label']:20s} → {pair['pred_label']:20s}: {pair['count']:3d} times")

print('\nDeiT-tiny - Top 5 Most Confused Pairs:')
print('-' * 60)
for pair in deit_confused:
    print(f"{pair['true_label']:20s} → {pair['pred_label']:20s}: {pair['count']:3d} times")

### Qualitative Analysis: ตัวอย่างรูปที่ทำนายถูกและผิด

การดูตัวอย่าง concrete ช่วยให้เข้าใจว่า model ทำงานได้ดีหรือไม่

In [ ]:
# ฟังก์ชันหาตัวอย่างรูปที่ทำนายถูก/ผิด
def find_examples(y_true, y_pred, viz_data, label_names, n_correct=5, n_wrong=5):
    """Find correctly and incorrectly predicted examples"""
    correct_indices = np.where(y_true == y_pred)[0]
    wrong_indices = np.where(y_true != y_pred)[0]

    # สุ่มเลือก
    correct_samples = np.random.choice(correct_indices, size=min(n_correct, len(correct_indices)), replace=False)
    wrong_samples = np.random.choice(wrong_indices, size=min(n_wrong, len(wrong_indices)), replace=False)

    results = {'correct': [], 'wrong': []}

    for idx in correct_samples:
        results['correct'].append({
            'image': viz_data[idx]['image'],
            'true_label': label_names[y_true[idx]],
            'pred_label': label_names[y_pred[idx]],
        })

    for idx in wrong_samples:
        results['wrong'].append({
            'image': viz_data[idx]['image'],
            'true_label': label_names[y_true[idx]],
            'pred_label': label_names[y_pred[idx]],
        })

    return results

resnet_examples = find_examples(resnet_y_true, resnet_y_pred, resnet_test_viz, label_list)
deit_examples = find_examples(deit_y_true, deit_y_pred, deit_test_viz, label_list)

In [ ]:
# แสดงตัวอย่างที่ทำนายถูก (ResNet-18)
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, example in enumerate(resnet_examples['correct']):
    axes[i].imshow(example['image'])
    axes[i].set_title(f"True: {example['true_label'][:15]}\nPred: {example['pred_label'][:15]}", fontsize=10)
    axes[i].axis('off')
plt.suptitle('ResNet-18: Correct Predictions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# แสดงตัวอย่างที่ทำนายผิด (ResNet-18)
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, example in enumerate(resnet_examples['wrong']):
    axes[i].imshow(example['image'])
    axes[i].set_title(f"True: {example['true_label'][:15]}\nPred: {example['pred_label'][:15]}", fontsize=10)
    axes[i].axis('off')
plt.suptitle('ResNet-18: Incorrect Predictions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# แสดงตัวอย่างที่ทำนายถูก (DeiT-tiny)
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, example in enumerate(deit_examples['correct']):
    axes[i].imshow(example['image'])
    axes[i].set_title(f"True: {example['true_label'][:15]}\nPred: {example['pred_label'][:15]}", fontsize=10)
    axes[i].axis('off')
plt.suptitle('DeiT-tiny: Correct Predictions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# แสดงตัวอย่างที่ทำนายผิด (DeiT-tiny)
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, example in enumerate(deit_examples['wrong']):
    axes[i].imshow(example['image'])
    axes[i].set_title(f"True: {example['true_label'][:15]}\nPred: {example['pred_label'][:15]}", fontsize=10)
    axes[i].axis('off')
plt.suptitle('DeiT-tiny: Incorrect Predictions', fontsize=14)
plt.tight_layout()
plt.show()

## 8. เปรียบเทียบและสรุป

สุดท้าย เราจะสรุปผลการเปรียบเทียบระหว่าง CNN (ResNet-18) กับ Vision Transformer (DeiT-tiny)

In [ ]:
# สร้างตารางเปรียบเทียบ
comparison_data = {
    'Model': ['ResNet-18 (CNN)', 'DeiT-tiny (ViT)'],
    'Parameters': [
        f"{sum(p.numel() for p in resnet_model.parameters()):,}",
        f"{sum(p.numel() for p in deit_model.parameters()):,}"
    ],
    'Training Time (min)': [
        f"{resnet_training_time/60:.1f}",
        f"{deit_training_time/60:.1f}"
    ],
    'Test Accuracy': [
        f"{resnet_test_metrics['test_accuracy']:.4f}",
        f"{deit_test_metrics['test_accuracy']:.4f}"
    ],
    'Test F1 (Macro)': [
        f"{resnet_test_metrics['test_f1_macro']:.4f}",
        f"{deit_test_metrics['test_f1_macro']:.4f}"
    ],
    'Test F1 (Weighted)': [
        f"{resnet_test_metrics['test_f1_weighted']:.4f}",
        f"{deit_test_metrics['test_f1_weighted']:.4f}"
    ],
    'Test Precision (Macro)': [
        f"{resnet_test_metrics['test_precision_macro']:.4f}",
        f"{deit_test_metrics['test_precision_macro']:.4f}"
    ],
    'Test Recall (Macro)': [
        f"{resnet_test_metrics['test_recall_macro']:.4f}",
        f"{deit_test_metrics['test_recall_macro']:.4f}"
    ],
}

comparison_df = pd.DataFrame(comparison_data)

# แสดงตาราง
from IPython.display import display
display(comparison_df.style.set_properties(**{'text-align': 'center'}).set_table_styles([{
    'selector': 'th',
    'props': [('background-color', '#f0f0f0'), ('color', 'black'), ('font-weight', 'bold')]
}]))

### สรุปและอภิปรายผลลัพธ์

#### สิ่งที่ควรสังเกต:

1. **Architecture Differences**
   - **ResNet-18 (CNN)**: ใช้ convolution layers ซึ่ง capture local patterns และ hierarchical features
   - **DeiT-tiny (ViT)**: ใช้ self-attention mechanism ซึ่ง capture global relationships ระหว่าง patches

2. **Model Size**
   - ResNet-18: ~11M parameters
   - DeiT-tiny: ~5M parameters (เล็กกว่าครึ่ง)

3. **Performance**
   - ดูจาก metrics ว่า model ไหนทำงานได้ดีกว่า
   - DeiT-tiny มักจะมีประสิทธิภาพดีกว่าแม้จะเล็กกว่า เพราะใช้ attention mechanisms
   - แต่ training ช้ากว่าเนื่องจาก computational complexity ของ self-attention

4. **Training Time**
   - CNNs (ResNet) มักจะเร็วกว่า เพราะ convolutions มี optimizations ดีๆ
   - ViTs มี computational overhead จาก attention mechanisms

5. **Error Patterns**
   - ดูจาก confusion matrices ว่าแต่ละ model สับสน classes ไหนบ้าง
   - บาง models อาจสับสน breeds ที่มีลักษณะคล้ายกัน (เช่น หมาพันธุ์ที่หน้าคล้ายกัน)

#### เมื่อไหร่ควรเลือกใช้ model ไหน?

- **เลือก ResNet (CNN) ถ้า:**
  - ต้องการ training speed
  - มีข้อมูลไม่มากนัก
  - ต้องการ model ที่ simple, interpretable

- **เลือก DeiT/ViT ถ้า:**
  - ต้องการ accuracy สูงสุด
  - มีข้อมูลมากพอ
  - ต้องการ model ที่เล็กกว่าแต่ performance ดี

#### สิ่งที่ได้เรียนรู้:

1. ใช้ HuggingFace `transformers`, `datasets`, `evaluate` libraries
2. สร้าง dataset จาก CSV metadata files
3. Fine-tune pre-trained models ด้วย `Trainer` API
4. Compare CNN vs Vision Transformer architectures
5. Evaluate models อย่างละเอียด (metrics, confusion matrices, qualitative analysis)

## 9. Gradio Demo - ทดลองใช้ Models แบบ Interactive

ตอนนี้เราจะสร้าง Gradio demo ที่ให้คุณ upload รูปและเลือกใช้งานได้ว่าจะใช้ ResNet-18 (CNN) หรือ DeiT-tiny (ViT) ในการทำนาย

In [ ]:
# ติดตั้ง Gradio
!pip install gradio -q

In [ ]:
# Import Gradio
import gradio as gr

# ย้าย models ไปยัง GPU ถ้ามี
if torch.cuda.is_available():
    resnet_model = resnet_model.to(device)
    deit_model = deit_model.to(device)
    resnet_model.eval()
    deit_model.eval()
    print('Models moved to GPU and set to eval mode')
else:
    print('Using CPU for inference')

# ฟังก์ชันสำหรับทำนายภาพ
def predict_image(image, model_choice):
    """
    ทำนาย label ของภาพโดยใช้ model ที่เลือก

    Args:
        image: รูปภาพที่ upload
        model_choice: 'ResNet-18 (CNN)' หรือ 'DeiT-tiny (ViT)'

    Returns:
        HTML output พร้อมผลการทำนาย
    """
    if image is None:
        return "<div style='text-align: center; color: #e74c3c; padding: 50px;'>Please upload an image!</div>"

    # เลือก model และ processor ตามที่ user เลือก
    if model_choice == 'ResNet-18 (CNN)':
        model = resnet_model
        processor = resnet_processor
    else:  # DeiT-tiny (ViT)
        model = deit_model
        processor = deit_processor

    # Preprocess image
    inputs = processor(images=image, return_tensors='pt')

    # Move to device
    if torch.cuda.is_available():
        inputs = {k: v.to(device) for k, v in inputs.items()}

    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # Get probabilities
    probs = torch.nn.functional.softmax(logits[0], dim=-1)

    # Get predicted label
    predicted_class_idx = probs.argmax().item()
    predicted_label = label_list[predicted_class_idx]
    confidence = probs[predicted_class_idx].item()

    # Get top 3 predictions
    top3_probs, top3_indices = torch.topk(probs, 3)

    # Create HTML for results
    result_html = """
    <div style="font-family: Arial, sans-serif;">
        <h3 style="color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;">
            Prediction Results
        </h3>

        <div style="margin: 15px 0; padding: 15px; background-color: #f39c1215; border-left: 4px solid #f39c12; border-radius: 4px;">
            <strong style="color: #f39c12; font-size: 18px;">Model:</strong> {}<br>
            <strong style="color: #f39c12; font-size: 18px;">Predicted Breed:</strong> {}<br>
            <strong style="color: #f39c12; font-size: 18px;">Confidence:</strong> {:.2f}%
        </div>

        <h4 style="color: #2c3e50; margin-top: 20px;">Top 3 Predictions:</h4>
    """.format(model_choice, predicted_label, confidence * 100)

    colors = ['#27ae60', '#3498db', '#95a5a6']  # Green, Blue, Gray

    for i, (prob, idx) in enumerate(zip(top3_probs, top3_indices)):
        label_name = label_list[idx.item()]
        percentage = prob.item() * 100
        color = colors[i]

        result_html += f"""
        <div style="margin: 10px 0; padding: 10px; background-color: {color}15; border-left: 4px solid {color}; border-radius: 4px;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 5px;">
                <strong style="color: {color}; font-size: 16px;">{i+1}. {label_name}</strong>
                <span style="color: {color}; font-weight: bold; font-size: 18px;">{percentage:.2f}%</span>
            </div>
            <div style="background-color: #ecf0f1; height: 8px; border-radius: 4px; overflow: hidden;">
                <div style="background-color: {color}; height: 100%; width: {percentage}%; transition: width 0.3s;"></div>
            </div>
        </div>
        """

    result_html += "</div>"

    return result_html

# สร้าง Gradio interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🐕🐈 Pet Breed Classification Demo

        อัปโหลดรูปสัตว์เลี้ยง (สุนัขหรือแมว) และเลือก model ที่ต้องการใช้ทำนายพันธุ์!

        ### Available Models:
        - **ResNet-18 (CNN)**: Convolutional Neural Network - เร็วกว่า เหมาะกับ real-time applications
        - **DeiT-tiny (ViT)**: Vision Transformer - มักจะแม่นยำกว่า ใช้ attention mechanisms
        """
    )

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                label="Upload Pet Image",
                type="pil",
                sources=["upload", "clipboard"],
                elem_id="image-upload"
            )

            model_choice = gr.Radio(
                choices=['ResNet-18 (CNN)', 'DeiT-tiny (ViT)'],
                value='ResNet-18 (CNN)',
                label="Select Model",
                info="Choose which model to use for prediction"
            )

            predict_btn = gr.Button(
                "🔮 Predict",
                variant="primary",
                size="lg"
            )

        with gr.Column():
            output_html = gr.HTML(
                label="Prediction Results",
                value="<div style='text-align: center; color: #7f8c8d; padding: 50px;'>Upload an image and click Predict to see results!</div>"
            )

    # Connect the function
    predict_btn.click(
        fn=predict_image,
        inputs=[image_input, model_choice],
        outputs=output_html
    )

    gr.Markdown(
        """
        ---

        ### 📊 ข้อมูลเพิ่มเติม

        **Dataset:** Oxford-IIIT Pet Dataset (37 classes ของพันธุ์สุนัขและแมว)

        **เมื่อไหร่ควรเลือก model ไหน?**
        - เลือก **ResNet-18** เมื่อต้องการความเร็วในการทำนาย
        - เลือก **DeiT-tiny** เมื่อต้องการความแม่นยำสูงสุด

        💡 **Tip:** ลองเปรียบเทียบผลลัพธ์จากทั้งสอง models กับรูปเดียวกัน!
        """
    )

# Launch the demo
print("Launching Gradio demo...")
demo.launch(share=False)

---

# 10. HuggingFace Datasets Library - Complete Tutorial

🎓 **Section 10 นี้เป็น tutorial แยกจาก main workshop** สอนการใช้งาน HuggingFace Datasets library อย่างละเอียดและเป็นระบบ

## เนื้อหาใน Section นี้:

1. **ทำไมต้องใช้ HuggingFace Datasets?** - ข้อดีและประโยชน์
2. **วิธีการโหลด Datasets** - จาก Hub, local files, Python objects
3. **สร้าง Dataset จาก Python Objects** - dict, list, pandas, numpy
4. **การจัดการ Datasets** - map, filter, shuffle, split
5. **DatasetDict** - จัดการหลาย splits พร้อมกัน
6. **Performance Tips** - ปรับปรุงความเร็วและการใช้ memory
7. **การบันทึกและแชร์** - save, load, push to Hub

---

### 10.1 ทำไมต้องใช้ HuggingFace Datasets?

**HuggingFace Datasets** เป็น library สำหรับจัดการ datasets ขนาดใหญ่อย่างมีประสิทธิภาพ:

- **ขนาดใหญ่**: มี datasets กว่า 10,000+ ชุด ใน Hub
- **ประหยัด Memory**: ใช้ memory-mapped files โหลด datasets ขนาด TB ได้
- **รวดเร็ว**: Built-in caching และ multiprocessing
- **ง่ายต่อการใช้งาน**: API ที่สะดวกและ consistent
- **Integration**: ทำงานร่วมกับ `transformers`, `evaluate` ได้ดี

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict, ClassLabel
import numpy as np
import pandas as pd
print('✅ HuggingFace Datasets library ready!')

### 10.2 วิธีการโหลด Datasets

มีหลายวิธีในการโหลด datasets ขึ้นอยู่กับรูปแบบข้อมูลที่คุณมี

In [ ]:
print('=== วิธีที่ 1: โหลดจาก HuggingFace Hub ===\n')
print('# Example: โหลด IMDb dataset (สำหรับ text classification)')
print('imdb_dataset = load_dataset("imdb")')
print('print(imdb_dataset)\n')
print('ข้อดี:')
print('  ✓ ไม่ต้อง download และ extract files เอง')
print('  ✓ ข้อมูลถูก organize และ document อย่างดี')
print('  ✓ มี version control และ citations พร้อม\n')

In [ ]:
print('=== วิธีที่ 2: โหลดจาก Local Files (สิ่งที่เราใช้ใน workshop) ===\n')
print('# CSV Files (สิ่งที่เราใช้):')
print('dataset = load_dataset("csv", data_files="path/to/file.csv")')
print('dataset = load_dataset("csv", data_files={"train": train.csv, "test": test.csv})')
print('')
print('# JSON Files:')
print('dataset = load_dataset("json", data_files="path/to/file.json")')
print('dataset = load_dataset("json", data_files="path/to/*.json")  # glob pattern')
print('')
print('# Text Files:')
print('dataset = load_dataset("text", data_files="path/to/file.txt")')
print('')
print('# Parquet Files (efficient for large datasets):')
print('dataset = load_dataset("parquet", data_files="path/to/file.parquet")')
print('')
print('# Image Folders:')
print('dataset = load_dataset("imagefolder", data_dir="path/to/images")')
print('# Structure: data_dir/class1/img1.jpg, class1/img2.jpg, ...\n')
print('💡 Tip: ทุกวิธีรองรับ data_files เป็น dictionary เพื่อโหลดหลาย splits พร้อมกัน!')

### 10.3 สร้าง Dataset จาก Python Objects

ถ้าคุณมีข้อมูลอยู่แล้วในรูปแบบ Python (list, dict, pandas DataFrame), คุณสามารถสร้าง Dataset ได้โดยตรง

In [ ]:
print('=== สร้าง Dataset จาก Python Objects ===\n')

print('1. จาก Dictionary:')
data_dict = {'text': ['Hello', 'World'], 'label': [0, 1]}
dataset_from_dict = Dataset.from_dict(data_dict)
print(f'   Dataset.from_dict() → {len(dataset_from_dict)} examples\n')

print('2. จาก List of Dictionaries:')
data_list = [{'text': 'Hello', 'label': 0}, {'text': 'World', 'label': 1}]
dataset_from_list = Dataset.from_list(data_list)
print(f'   Dataset.from_list() → {len(dataset_from_list)} examples\n')

print('3. จาก Pandas DataFrame:')
df = pd.DataFrame({'text': ['Hello', 'World'], 'label': [0, 1]})
dataset_from_df = Dataset.from_pandas(df)
print(f'   Dataset.from_pandas() → {len(dataset_from_df)} examples\n')

print('💡 Tip: Dataset.from_dict() คือวิธีที่ flexible ที่สุด!')

### 10.4 การจัดการและ Transform Datasets

HuggingFace Datasets มี methods ที่ทรงพลังสำหรับการจัดการข้อมูล

In [ ]:
# สร้าง dataset ตัวอย่าง
sample_data = {'text': ['Hello', 'World', 'Hi'], 'label': [0, 1, 0]}
sample_dataset = Dataset.from_dict(sample_data)

print('=== Dataset Operations ===\n')

print('1. Slicing - เลือกบาง examples:')
print(f'   dataset[0] = {sample_dataset[0]}')
print(f'   dataset["text"] = {sample_dataset["text"]}\n')

print('2. Filter - กรองข้อมูล:')
def filter_label_0(example):
    return example['label'] == 0

filtered = sample_dataset.filter(filter_label_0)
print(f'   Filter label=0: {len(filtered)} examples\n')

print('3. Map - แปลงข้อมูล (method ที่ทรงพลังที่สุด):')
def add_length(example):
    example['length'] = len(example['text'])
    return example

mapped = sample_dataset.map(add_length)
print(f'   Added "length" column: {mapped.column_names}\n')

print('4. Cast - แปลงประเภท features:')
class_label = ClassLabel(num_classes=2, names=['neg', 'pos'])
casted = sample_dataset.cast_column('label', class_label)
print(f'   Label type: {casted.features["label"]}\n')

print('5. Shuffle - สลับลำดับ:')
shuffled = sample_dataset.shuffle(seed=42)
print(f'   First example: {shuffled[0]["text"]}\n')

print('6. Split - แบ่ง train/test:')
split = sample_dataset.train_test_split(test_size=0.5, seed=42)
print(f'   Train: {len(split["train"])}, Test: {len(split["test"])}\n')

print('💡 Tip: map() ใช้สำหรับ preprocessing, tokenization, augmentation ฯลฯ!')

### 10.5 DatasetDict: จัดการหลาย Splits พร้อมกัน

`DatasetDict` ใช้สำหรับจัดการหลาย splits (train, validation, test) ใน object เดียว

In [ ]:
print('=== DatasetDict - จัดการหลาย splits ใน object เดียว ===\n')

# สร้าง DatasetDict
dataset_dict_example = DatasetDict({
    'train': Dataset.from_dict({'text': ['A', 'B'], 'label': [0, 1]}),
    'test': Dataset.from_dict({'text': ['C'], 'label': [0]}),
})

print(f'DatasetDict: {list(dataset_dict_example.keys())}')
print(f'Train size: {len(dataset_dict_example["train"])}')
print(f'Test size: {len(dataset_dict_example["test"])}\n')

print('Operations กับทุก splits พร้อมกัน:')

# Map function ไปทุก splits
def add_length_all(example):
    example['length'] = len(example['text'])
    return example

dataset_dict_with_length = dataset_dict_example.map(add_length_all)
print(f'✓ map() ถูก apply กับทุก splits\n')

# Filter ทุก splits
def filter_label_0_all(example):
    return example['label'] == 0

filtered_dict = dataset_dict_example.filter(filter_label_0_all)
print(f'✓ filter() ถูก apply กับทุก splits\n')

# Shuffle ทุก splits
shuffled_dict = dataset_dict_example.shuffle(seed=42)
print(f'✓ shuffle() ถูก apply กับทุก splits\n')

print('💡 Tip: DatasetDict เหมาะสำหรับเก็บ train/val/test splits ไว้ด้วยกัน!')
print(f'   เราใช้มันใน workshop: {list(dataset_dict.keys())}')

### 10.6 Performance Tips & Saving

เทคนิคในการให้งานเร็วขึ้นและการบันทึก datasets

In [ ]:
print('=== Performance Tips ===\n')

print('1. ใช้ Multiprocessing (num_proc):')
print('   dataset.map(func, num_proc=4)')
print('   → ประมวลผลแบบ parallel ด้วย 4 workers\n')

print('2. Automatic Caching:')
print('   • ผลลัพธ์จาก map() จะถูก cache อัตโนมัติ')
print('   • รันครั้งต่อไปจะเร็วมาก!')
print('   • Cache อยู่ใน ~/.cache/huggingface/datasets/\n')

print('3. Set Format สำหรับ PyTorch/TensorFlow:')
print('   dataset.set_format("torch", columns=["input_ids", "label"])')
print('   → แปลง columns เหล่านี้เป็น torch tensors อัตโนมัติ\n')

print('4. Arrow Format - ประหยัด Memory:')
print('   • ใช้ Apache Arrow format')
print('   • Memory-mapped files - ไม่โหลดทั้งหมดเข้า RAM')
print('   • สามารถโหลด datasets ขนาด TB ได้!\n')

print('=== Saving & Sharing ===\n')

print('1. Save to disk:')
print('   dataset.save_to_disk("path/to/dataset")')

print('2. Load from disk:')
print('   dataset = load_from_disk("path/to/dataset")')

print('3. Push to HuggingFace Hub:')
print('   dataset.push_to_hub("username/dataset-name")')
print('   → อัปโหลดไป Hub แชร์ให้คนอื่นใช้\n')

print('💡 Tip: ใช้ num_proc=4-8 เมื่อทำ heavy processing!')